# Sistem CBR Putusan Pengadilan Berbasis Python

Notebook ini menyatukan seluruh alur Case-Based Reasoning untuk analisis putusan pengadilan. Alur mengikuti lima tahap: build case base, case representation, case retrieval, solution reuse, dan evaluation.

Dataset proyek ini menggunakan putusan pidana khusus ITE dari Direktori Putusan Mahkamah Agung. Sistem memakai TF-IDF dan cosine similarity sebagai baseline retrieval, lalu memakai weighted vote top-k untuk label solusi.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

Project root: d:\KULIAH S1\SEMESTER 6\penalaran komputer\SUB CPMK 3\cbr-putusan-ite-pn-surabaya\cbr-putusan-ite-pn-surabaya


In [2]:
import json
import pandas as pd
from IPython.display import display

from src.config import RAW_DOCS_DIR, RAW_TEXT_DIR, CASES_CSV, QUERIES_JSON
from src.preprocess import iter_source_files, process_raw_documents
from src.extract_metadata import build_case_representation
from src.retrieval import fit_retrieval_model, retrieve
from src.reuse import predict_outcome
from src.evaluation import evaluate_retrieval, evaluate_prediction, build_failure_analysis

## 1. Build Case Base

In [3]:
source_files = iter_source_files(RAW_DOCS_DIR)
print("Jumlah dokumen sumber:", len(source_files))
print("Memenuhi minimal 30 dokumen:", len(source_files) >= 30)

FORCE_REPROCESS = False
source_map_path = ROOT / "data" / "processed" / "source_files.csv"
existing_raw = sorted(RAW_TEXT_DIR.glob("case_*.txt"))

if FORCE_REPROCESS or not source_map_path.exists() or len(existing_raw) < 30:
    source_df = pd.DataFrame(process_raw_documents(RAW_DOCS_DIR, RAW_TEXT_DIR))
else:
    source_df = pd.read_csv(source_map_path)

display(source_df.head())
quality_summary = pd.DataFrame([
    {"metric": "jumlah_dokumen", "value": len(source_df)},
    {"metric": "dokumen_terlalu_pendek", "value": int((source_df["word_count"] < 100).sum())},
    {"metric": "rata_rata_jumlah_kata", "value": round(float(source_df["word_count"].mean()), 2)},
])
display(quality_summary)

Jumlah dokumen sumber: 50
Memenuhi minimal 30 dokumen: True


,case_id,source_file,raw_text_file,word_count,status
0,case_001,putusan_1085_pid.sus_2020_pt_sby_2026062414570...,raw\case_001.txt,1301,ok
1,case_002,putusan_1111_pid.sus_2020_pt_sby_2026062414573...,raw\case_002.txt,2782,ok
2,case_003,putusan_113_pid.sus_2022_pt_sby_20260624145811...,raw\case_003.txt,948,ok
3,case_004,putusan_113_pid.sus_2022_pt_sby_20260624145811...,raw\case_004.txt,948,ok
4,case_005,putusan_1184_pid.sus_2020_pt_sby_2026062414582...,raw\case_005.txt,1147,ok


,metric,value
0,jumlah_dokumen,50.0
1,dokumen_terlalu_pendek,1.0
2,rata_rata_jumlah_kata,1904.2


## 2. Case Representation

In [4]:
cases = build_case_representation()
print("Jumlah kasus:", len(cases))
display(cases[["case_id", "no_perkara", "tanggal_putusan", "pengadilan", "jenis_perkara", "pihak", "label_putusan", "word_count"]].head(10))
display(cases["label_putusan"].value_counts().rename_axis("label_putusan").reset_index(name="jumlah"))

d:\KULIAH S1\SEMESTER 6\penalaran komputer\SUB CPMK 3\cbr-putusan-ite-pn-surabaya\cbr-putusan-ite-pn-surabaya\src\extract_metadata.py:295: UserWarning: Cell contents too long (37729), truncated to 32767 characters
  df.to_excel(PROCESSED_DIR / "cases.xlsx", index=False)
d:\KULIAH S1\SEMESTER 6\penalaran komputer\SUB CPMK 3\cbr-putusan-ite-pn-surabaya\cbr-putusan-ite-pn-surabaya\src\extract_metadata.py:295: UserWarning: Cell contents too long (37149), truncated to 32767 characters
  df.to_excel(PROCESSED_DIR / "cases.xlsx", index=False)
d:\KULIAH S1\SEMESTER 6\penalaran komputer\SUB CPMK 3\cbr-putusan-ite-pn-surabaya\cbr-putusan-ite-pn-surabaya\src\extract_metadata.py:295: UserWarning: Cell contents too long (40225), truncated to 32767 characters
  df.to_excel(PROCESSED_DIR / "cases.xlsx", index=False)


Jumlah kasus: 50


,case_id,no_perkara,tanggal_putusan,pengadilan,jenis_perkara,pihak,label_putusan,word_count
0,case_001,1085/PID.SUS/2020/PT SBY,17 September 2020,Pengadilan Tinggi Surabaya,Pidana Khusus ITE,BAGUS ARIYANTO WIBOWO,menguatkan,1301
1,case_002,1111/PID.SUS/2020/PT.SBY,15 September 2020,Pengadilan Tinggi Surabaya,Pidana Khusus ITE,FAUZIA DONY KURNIAWAN bin SUDARSONO,pidana penjara,2782
2,case_003,113/PID.SUS/2022/PT SBY,21 Februari 2022,Pengadilan Tinggi Surabaya,Pidana Khusus ITE,SYAIFUL IMAM,pidana penjara,948
3,case_004,113/PID.SUS/2022/PT SBY,21 Februari 2022,Pengadilan Tinggi Surabaya,Pidana Khusus ITE,SYAIFUL IMAM,pidana penjara,948
4,case_005,1184/PID.SUS/2020/PT SBY,21 Oktober 2020,Pengadilan Tinggi Surabaya,Pidana Khusus ITE,MARIA ULFA,pidana penjara,1147
5,case_006,1186/PID.SUS/2024/PT SBY,02 Mei 2024,Pengadilan Tinggi Surabaya,Pidana Khusus ITE,ARJUN NADIKA SYAHID bin KASMIN,pidana penjara,1951
6,case_007,1194/PID.SUS/2022/PT SBY,5 Januari 2023,Pengadilan Tinggi Surabaya,Pidana Khusus ITE,"ISLAH JIWA WILSMAN,S.T",bebas,4549
7,case_008,1208/PID.SUS/2024/PT SBY,04 Juni 2023,Pengadilan Tinggi Surabaya,Pidana Khusus ITE,Indra Setiawati binti Alm Sumarto,lepas,1784
8,case_009,1214/PID.SUS/2024/PT SBY,8 Oktober 2024,Pengadilan Tinggi Surabaya,Pidana Khusus ITE,Dimas Eka Rhomdhoni bin Sugoto,pidana penjara,2358
9,case_010,,28 Agustus 2024,Pengadilan Tinggi Surabaya,Pidana Khusus ITE,MUHAMMAD SOBRI,pidana penjara,1724


,label_putusan,jumlah
0,pidana penjara,34
1,bebas,7
2,menguatkan,4
3,tidak diketahui,4
4,lepas,1


## 3. Case Retrieval

In [5]:
model_info = fit_retrieval_model()
model_info

{'num_cases': 50,
 'num_features': 29884,
 'num_labels': 5,
 'label_distribution': {'pidana penjara': 34,
  'bebas': 7,
  'menguatkan': 4,
  'tidak diketahui': 4,
  'lepas': 1}}

In [6]:
query = "terdakwa mendistribusikan informasi elektronik bermuatan penghinaan dan pencemaran nama baik melalui media sosial"
top_k = retrieve(query, k=5)
display(top_k[["rank", "case_id", "similarity", "no_perkara", "pihak", "label_putusan"]])

,rank,case_id,similarity,no_perkara,pihak,label_putusan
0,1,case_005,0.063356,1184/PID.SUS/2020/PT SBY,MARIA ULFA,pidana penjara
1,2,case_022,0.056747,209/PID.SUS/2024/PN PMK,Terdakwa,bebas
2,3,case_008,0.053700,1208/PID.SUS/2024/PT SBY,Indra Setiawati binti Alm Sumarto,lepas
3,4,case_028,0.052750,26/PID.SUS/2022/PT SBY,"Totok Dwihartono, SH bin Saman Sugondo",pidana penjara
4,5,case_017,0.047496,1381/PID.SUS/2023/PT SBY,Ramli bin Umar,pidana penjara


## 4. Solution Reuse

In [7]:
prediction = predict_outcome(query, k=5)
print("Predicted label:", prediction["predicted_label"])
print("Top-1 case:", prediction["top_1_case_id"], prediction["top_1_no_perkara"])
print("Similarity:", prediction["top_1_similarity"])
print("\nSolusi dari amar putusan top-1:\n")
print(prediction["predicted_solution"][:1500])

Predicted label: pidana penjara
Top-1 case: case_005 1184/PID.SUS/2020/PT SBY
Similarity: 0.063356

Solusi dari amar putusan top-1:

MENGADILI:-Menerima permintaan banding dari Terdakwa dan Penuntut Umumtersebut; -Menguatkan PUTUSAN Pengadilan Negeri Banyuwangi, tanggal04 Agustus 2020 Nomor 206/Pid.Sus/2020/PN. Byw; Halaman 9 PUTUSAN Perkara Nomor 1184/PID.SUS/2020/PT SBY


## 5. Evaluation

In [8]:
queries = pd.DataFrame(json.loads(QUERIES_JSON.read_text(encoding="utf-8")))
display(queries)

retrieval_metrics = evaluate_retrieval(k=5)
prediction_metrics = evaluate_prediction(k=5)
failure_analysis = build_failure_analysis(retrieval_metrics, prediction_metrics)

display(retrieval_metrics)
display(prediction_metrics)
display(failure_analysis)

,query_id,query,ground_truth_case_id,expected_label
0,q1,terdakwa menyebarkan konten penghinaan melalui...,case_001,pidana penjara
1,q2,terdakwa menyebarkan informasi elektronik yang...,case_002,pidana penjara


,query_id,query,ground_truth_case_id,top_1_case_id,top_k_case_ids,hit_at_k,top_1_correct,precision_at_k,recall_at_k,f1_at_k
0,q1,terdakwa menyebarkan konten penghinaan melalui...,case_001,case_014,"case_014,case_003,case_004,case_022,case_029",0.0,0.0,0.0,0.0,0.0
1,q2,terdakwa menyebarkan informasi elektronik yang...,case_002,case_005,"case_005,case_017,case_013,case_022,case_018",0.0,0.0,0.0,0.0,0.0
2,SUMMARY,,,,,0.0,0.0,0.0,0.0,0.0


,query_id,query,expected_label,predicted_label,predicted_label_classifier,top_1_case_id,top_1_similarity,correct,precision_macro,recall_macro,f1_macro
0,q1,terdakwa menyebarkan konten penghinaan melalui...,pidana penjara,pidana penjara,pidana penjara,case_014,0.040302,1.0,NaN,NaN,NaN
1,q2,terdakwa menyebarkan informasi elektronik yang...,pidana penjara,pidana penjara,pidana penjara,case_005,0.047623,1.0,NaN,NaN,NaN
2,SUMMARY,,,,,,,1.0,1.0,1.0,1.0


,query_id,failure_type,possible_cause,recommendation
0,q1,ground truth tidak masuk top-k; top-1 berbeda ...,"query terlalu pendek, variasi istilah hukum, m...","tambah query uji berlabel, bersihkan OCR/PDF l..."
1,q2,ground truth tidak masuk top-k; top-1 berbeda ...,"query terlalu pendek, variasi istilah hukum, m...","tambah query uji berlabel, bersihkan OCR/PDF l..."


## Kesimpulan

Notebook ini menunjukkan bahwa sistem CBR sederhana dapat membangun case base, mengekstrak representasi kasus, melakukan retrieval, mengambil solusi dari amar putusan, dan mengevaluasi hasil dengan metrik dasar. Sistem ini layak sebagai baseline akademik. Pengembangan berikutnya dapat memakai embedding IndoBERT, validasi label oleh ahli hukum, dan evaluasi dengan query yang lebih banyak.